# RotQuant Qwen3.5-4B — focused decode validation after cache repair

## Goal
Select **A100 40GB**, check the saved source path, then **Runtime → Run all**.
Keep the existing **W5/scale8 + W6 vocabulary** weights unchanged.

**followup2 completed the BF16 and Unsloth UD-Q4 baselines** and the
public/private conventional preflight. Its decode4 numerical cases
passed, but a dispatch-counter gate stopped before candidate model tests.
A Linux CPU regression reproduced the bug: restored library aliases
could open a second instance with zero counters. Diagnostics now bind
through the execution library's dependencies, and a cheap probe checks
fresh counter increments **before model export or timing**.

Do not repeat the completed baseline downloads/timings by default.
Run fresh reference/candidate pairs for the opt-in **decode4** kernel:
one warp per output row, shared codebook,
hoisted group scales and the same floating-point reduction order.
It retains tiled4 for prefill. **No decode speedup or CUDA correctness
is claimed yet.** Exact operator and bounded retained-model numerical
gates precede timing; their thresholds are unchanged.
No calibration, requantization, adapters or training are run.

## Setup — explicit controls
Keep W6 alone for the first pilot. W8 is supported as a separate follow-up:
set `ARMS = ("b5_v8_s0",)` and use a new run name when ready.

`SOURCE_ROOT` must contain the original checkpoint, `prepared.json`,
`preparation.json`, and `packed_probes.safetensors` for each arm.
A reports-only ZIP is insufficient. The source is never modified.

The default **90 active-minute** allowance includes dependency setup,
hashing/copying, build and tests. Performance caps scale with context and
repetitions (default **4 / 6 / 12 minutes**). The 2 tok/s and 16 GiB limits are spending
guards, not pass marks for quality or production readiness.

In [ ]:
from pathlib import Path
import json, re, shutil, subprocess, sys

REPO_REF = "main"  # resolved once; use the published full commit for repeat runs
SOURCE_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_packed_validation/8f10ee60fc7f")
CACHE_ROOT = Path("/content/drive/MyDrive/rotquant/native_artifact_cache/v1")
ARMS = ("b5_v6_s0",)
CONTEXTS = (128, 512)  # use (2048,) for a targeted rerun with a new RUN_NAME
RUN_NAME = "followup3"
ACTIVE_BUDGET_MINUTES = 90
BUILD_JOBS = 2
DECODE_STEPS = 32
MEASURED_REPETITIONS = 3  # plus one excluded warmup per context
MIN_DECODE_TOKENS_PER_SECOND = 2.0
MAX_PROCESS_VRAM_MIB = 16384

assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,63}", RUN_NAME)
assert 1 <= ACTIVE_BUDGET_MINUTES <= 180 and 1 <= BUILD_JOBS <= 32
assert ARMS and len(set(ARMS)) == len(ARMS) and set(ARMS) <= {"b5_v6_s0", "b5_v8_s0"}
BASELINES = ()  # () explicitly skips these downloads/runs
RUN_PROFILE = False
assert len(set(BASELINES)) == len(BASELINES) and set(BASELINES) <= {"bf16", "ud_q4"}

CANDIDATE = "decode4"  # "none" runs only fresh reference + BF16/UD-Q4
assert CANDIDATE in ("decode4", "none")
assert CANDIDATE != "none" or (BASELINES and not RUN_PROFILE)


### Connect Drive and pin the repository
This uses Colab's normal Drive mount; local `gws` authentication is not
required. The managed venv preserves Colab CUDA Torch and does not need
`ensurepip`, an external Hadamard kernel or `llama-cpp-python`.

Keep `CACHE_ROOT` private. Hashes detect accidental corruption, **not**
a malicious replacement of both cached binaries and their manifest.
Do not use downloaded/shared caches from an untrusted party.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert shutil.which("nvidia-smi") and shutil.which("nvcc"), "Select a CUDA GPU runtime."
subprocess.run(["nvidia-smi"], check=True, timeout=30)
for arm in ARMS:
    for item in ("checkpoint", "prepared.json", "preparation.json", "packed_probes.safetensors"):
        assert (SOURCE_ROOT / arm / item).exists(), f"Missing original evidence: {SOURCE_ROOT / arm / item}"

REPO_DIR = Path("/content/rotquant-native-followup/repository")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/CodeHalwell/rotquant.git", str(REPO_DIR)], check=True, timeout=300)
assert not subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip(), "Repository has edits; use a fresh runtime. No automatic reset."
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True, timeout=120)
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "FETCH_HEAD"], text=True).strip()
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", COMMIT], check=True, timeout=30)
for name in ("run_native_gpu_validation.py", "native_gpu_cache.py", "run_rq3_performance_pilot.py", "native_pilot_controls.py", "native_performance_study.py", "native_cuda_diagnostics.py", "run_native_gguf_baseline.py"):
    assert (REPO_DIR / "scripts" / name).exists(), "Pilot code is not present at REPO_REF. Stop before spending on the build."
sys.path.insert(0, str(REPO_DIR))
from scripts.native_pilot_controls import selected_contexts, validate_controls
CONTEXTS = selected_contexts(CONTEXTS)
validate_controls(128, DECODE_STEPS, MEASURED_REPETITIONS,
                  MIN_DECODE_TOKENS_PER_SECOND, MAX_PROCESS_VRAM_MIB)
WORK_DIR = Path("/content/rotquant-native-followup/work") / COMMIT[:12]
RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/native_gpu_followup") / COMMIT[:12] / RUN_NAME
print({"commit": COMMIT, "results": str(RESULT_ROOT), "cache": str(CACHE_ROOT),
       "arms": ARMS, "active_minutes": ACTIVE_BUDGET_MINUTES,
       "contexts": CONTEXTS, "decode_steps": DECODE_STEPS})

## Steps and checks
1. Verify saved evidence; build/restore the hash-checked native library.
2. **Early dispatch preflight:** prove one-token decode and four-token
   prefill increment the intended counters, and match reference outputs
   exactly. The report identifies the actual diagnostic library provider.
   Missing/wrong dispatch stops immediately; it is never waived.
3. Fresh RQ3 operators, W6/W8 tiny-model, conversion and saved-model parity.
4. Fresh reference timings at **128 and 512 tokens**, three measurements
   plus one excluded warmup, on this runtime/GPU.
5. With `CANDIDATE = "decode4"`: test operator arithmetic, partial row
   groups, permutations, long reduction widths, tiny models and retained
   parity, then collect matched candidate timing. The default kernel is
   unchanged. A failed gate stops execution, never relaxes thresholds.

The completed baseline reports stay historical evidence. They are not
imported as fresh timings or mixed into this run's speed ratios.
To explicitly repeat baselines, set `BASELINES = ("bf16", "ud_q4")`.
That also repeats the conventional public/private preflight first.
For baseline-only work set those baselines, `CANDIDATE = "none"` and
`RUN_PROFILE = False`. Fresh reference timings still run.
Profiles and 2,048-token measurements are skipped by default because the
previous study already answered those questions. You can explicitly add
2048 to `CONTEXTS` or set `RUN_PROFILE = True` with a candidate and a new
run name. Controls cannot change inside an existing result directory.

This diagnostic repair leaves native kernels and the binary-cache key
unchanged from **2c4037e76f71**. Its cached CUDA build can be restored
when hardware/toolchain checks match. No new CUDA compilation is required
solely for this repair.
A cache miss still needs a cold build (~27 minutes previously).
Original weights are reused, never trained or requantized.
Baseline downloads (~11.34 GB) are skipped by default. The private Drive
cache and original results are retained; don't clear them.
Every phase prints a persistent log, cap and heartbeat; every repetition
prints timing and progress. The **90 active-minute** allowance and phase
caps stop subprocesses, **not Colab billing**.

After any stop, run Results and Download; do not add manual repair cells.
Disconnect/delete the GPU runtime when finished. Never update a running
checkout. Local validation uses mocks/CPU and cannot establish CUDA
performance; real execution and rendered outputs must be checked here.

In [ ]:
from scripts.colab_runtime import run_live
command = [sys.executable, "-u", str(REPO_DIR / "scripts/run_native_gpu_validation.py"),
           "--output-dir", str(RESULT_ROOT), "--work-dir", str(WORK_DIR),
           "--source-root", str(SOURCE_ROOT), "--persistent-cache-dir", str(CACHE_ROOT),
           "--active-minutes", str(ACTIVE_BUDGET_MINUTES), "--jobs", str(BUILD_JOBS),
           "--performance-study", "--decode-steps", str(DECODE_STEPS),
           "--repetitions", str(MEASURED_REPETITIONS),
           "--min-decode-tps", str(MIN_DECODE_TOKENS_PER_SECOND),
           "--max-vram-mib", str(MAX_PROCESS_VRAM_MIB)]
for arm in ARMS:
    command.extend(["--arm", arm])
for context in CONTEXTS:
    command.extend(["--context", str(context)])
command.extend(["--study-candidate", CANDIDATE])
if not RUN_PROFILE:
    command.append("--skip-profile")
if not BASELINES:
    command.append("--no-baselines")
for baseline in BASELINES:
    command.extend(["--baseline", baseline])
run_live(command, "native-followup", repo_dir=REPO_DIR,
         log_root=RESULT_ROOT / "launch-logs")

## Results — also run after a stop
Throughput and diagnostic profiles are shown separately. Warmup is
excluded; VRAM is sampled process allocation, not an exact transient peak.
Profiles synchronize CUDA events per custom operation with graphs
disabled. **Do not compare their wall rates to normal throughput.**
Non-custom attention/SSM kernels, host work and copy costs are not
individually attributed by this first profiler; event times do not sum
to complete model wall time.

Reference timing generates greedy decode IDs; candidate and conventional
controls replay those exact IDs, while still performing the same argmax
and model calls. This isolates shape/token-path differences, not output
quality. Token-ID maps are checked without retokenizing multilingual text.
The GGUF controls are **not matched for quality or exact size**. Their
reported size is text-GGUF bytes; the RotQuant non-text sidecar is not in VRAM.

`comparison-*.json` contains ratios only for complete matched,
uninstrumented runs. No automatic kernel or recipe promotion occurs.

BF16/Unsloth timings are intentionally absent with `BASELINES = ()`. See the
archived followup2 results in `research/results/native_followup_2026_09_13/`.
Only fresh matched reference/candidate pairs can produce this run's speed ratios.

The conventional preflight summary below separates **required bridge gates**
from cross-backend numerical diagnostics. A Q4_0 diagnostic `False` is not
silently changed to `True`; BF16 still requires those numerical limits.
These tiny shared-library tests do not validate upstream kernels independently.


In [ ]:
from IPython.display import Markdown, display
from scripts.native_performance_summary import tables
summary_path = RESULT_ROOT / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("Workflow:", summary["status"], "| active minutes:", round(summary["active_minutes"], 1))
    for stage in summary["stages"]:
        print(stage["name"], stage["status"], f"{stage.get('elapsed_seconds', 0):.1f}s")
    if summary.get("error"):
        print("Stopped:", summary["error"])
timing, profiling = tables(RESULT_ROOT)
display(Markdown("### Uninstrumented throughput\n\n" + timing))
display(Markdown("### Diagnostic profiles — not throughput\n\n" + profiling))
archives = sorted(RESULT_ROOT.parent.glob(RUN_NAME + "-reports-*.zip"))
if archives:
    REPORTS_ZIP = archives[-1]
    print("Reports ZIP:", REPORTS_ZIP)
print("DISCONNECT AND DELETE the GPU runtime after downloading your reports.")

for path in sorted(RESULT_ROOT.glob("stages/conventional-preflight-*/attempt-*/report.json")):
    report = json.loads(path.read_text())
    print("Conventional preflight:", path.parent.parent.name, path.parent.name,
          {k: report.get(k) for k in ("format", "passed", "gpu_executed", "cross_backend_numerics_role", "error")})
    for device, result in report.get("same_backend", {}).items():
        print("  Required private/public agreement", device, result["passed"], result["metrics"])
    for caller, result in report.get("cross_backend", {}).items():
        print("  CPU/GPU", caller, "original numerical + trace limits passed:", result["passed"], result["metrics"])


## Download and next steps
Download the reports ZIP below, then disconnect/delete the GPU runtime.
Bring back the reports before changing more algorithms or running tasks.
We need measured bottleneck attribution, unchanged parity and a useful
uninstrumented improvement before enabling an optimised kernel by default.

In [ ]:
from google.colab import files
if "REPORTS_ZIP" in globals():
    files.download(str(REPORTS_ZIP))
else:
    print("No report archive found. Run the Results cell first.")